Import Required Libraries


In [ ]:
import numpy as np
import tensorflow as tf
import random

seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)

In [ ]:
!pip install lifelines
!pip install gseapy
!pip install decoupler
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import gseapy as gp
import time
import textwrap
import decoupler as dc

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from sklearn.decomposition import PCA

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, confusion_matrix

from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests

Data Loading (Gene Expression)

In [ ]:
rna_url = "https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/TCGA.PRAD.sampleMap/HiSeqV2.gz"

rna = pd.read_csv(rna_url, sep="\t", index_col=0)
print("Original RNA shape:", rna.shape)

Data Pre Processing (Gene Expression)

In [ ]:
rna = rna.T
rna.index = rna.index.str[:12]
rna = rna[~rna.index.duplicated(keep="first")]
print("RNA after transpose:", rna.shape)

In [ ]:
threshold = int(0.2 * rna.shape[0])
rna = rna.loc[:, (rna > 0).sum(axis=0) > threshold]
print("After zero filtering:", rna.shape)

In [ ]:
gene_variance = rna.var(axis=0)
sorted_variance = (gene_variance.sort_values(ascending=False).values)

plt.figure(figsize=(7, 5))

plt.plot(sorted_variance,linewidth=2,color="#4C72B0")

plt.axhline(y=1.0,linestyle="--",linewidth=2,color="#9A9A9A",label="Variance threshold = 1.0")

plt.xlabel("Gene Rank")
plt.ylabel("Gene Variance")
plt.title("Variance Distribution of Genes")

plt.grid(axis='y',alpha=0.08)
plt.ylim(0,max(sorted_variance) * 1.05)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
threshold = 1.0
selected_genes = gene_variance[gene_variance > threshold]
print("Number of genes with variance > 1.0:", len(selected_genes))
rna = rna[selected_genes.index]
print(rna.shape)
rna.head()

Data Loading & Data Pre Processing (Survival Data)

In [ ]:
clinical_url = "https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/TCGA.PRAD.sampleMap%2FPRAD_clinicalMatrix"
clinical = pd.read_csv(clinical_url, sep="\t", index_col=0)
clinical["PatientID"] = clinical.index.str[:12]
clinical = clinical.set_index("PatientID")
clinical = clinical.groupby(clinical.index).first()
print("Clinical shape:", clinical.shape)

In [ ]:
def yesno(x):
    if str(x).upper()=="YES":
        return 1
    if str(x).upper()=="NO":
        return 0
    return np.nan

clinical["rec1"] = clinical["biochemical_recurrence"].apply(yesno)
clinical["rec2"] = clinical["new_tumor_event_after_initial_treatment"].apply(yesno)
clinical["rec3"] = clinical["tumor_progression_post_ht"].apply(yesno)
clinical["recurrence"] = clinical[["rec1","rec2","rec3"]].max(axis=1)
print("Recurrence events:", clinical["recurrence"].sum())

In [ ]:
t1 = pd.to_numeric(clinical["days_to_first_biochemical_recurrence"], errors="coerce")
t2 = pd.to_numeric(clinical["days_to_new_tumor_event_after_initial_treatment"], errors="coerce")
t3 = pd.to_numeric(clinical["days_to_last_followup"], errors="coerce")

event_time = pd.concat([t1, t2], axis=1).min(axis=1)
event_time = event_time.fillna(t3)

clinical["time"] = np.where(clinical["recurrence"] == 1,event_time,t3)

clinical = clinical.dropna(subset=["time","recurrence"])
clinical = clinical[clinical["time"] > 0]

In [ ]:
print(clinical.index.nunique(), len(clinical))

Align RNA and Survival data

In [ ]:
common = rna.index.intersection(clinical.index)
print("Number of common patients:", len(common))
rna_final = rna.loc[common].copy()
survival = clinical.loc[common, ["time","recurrence"]].copy()
print("RNA final shape:", rna_final.shape)
print("Survival shape:", survival.shape)

In [ ]:
assert all(rna_final.index == survival.index)

Latent Feature Extraction Using Autoencoder

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
K.clear_session()

scaler = StandardScaler()
X_full = scaler.fit_transform(rna_final)

input_dim = X_full.shape[1]
inp = Input(shape=(input_dim,))
x = Dense(256, activation='tanh',kernel_regularizer=regularizers.l2(1e-4))(inp)
bottleneck = Dense(100, activation='tanh')(x)
x = Dense(256, activation='tanh')(bottleneck)
out = Dense(input_dim, activation='linear')(x)
autoencoder = Model(inp, out)
encoder = Model(inp, bottleneck)
autoencoder.compile(optimizer='adam', loss='mse')
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = autoencoder.fit(
    X_full, X_full,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

Z = encoder.predict(X_full, verbose=0)
latent_df = pd.DataFrame(Z, index=rna_final.index)

cox_df = latent_df.copy()
cox_df["time"] = survival["time"].values
cox_df["event"] = survival["recurrence"].values

cph = CoxPHFitter(penalizer=0.1)
cph.fit(cox_df, duration_col="time", event_col="event")

print("\nFinal Cox Model Trained")

encoder.save("final_encoder.h5")

print("Models saved successfully")

In [ ]:
plt.figure(figsize=(6,4))

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Autoencoder Training Loss')

plt.legend(loc="upper right", frameon=False)
plt.grid(alpha=0.08)

plt.tight_layout()
plt.show()

In [ ]:
autoencoder.summary()

Survival Modeling


In [ ]:
pvals = []
valid_cols = []

for col in latent_df.columns:

    df = pd.DataFrame({
        col: latent_df[col],
        "time": survival["time"],
        "event": survival["recurrence"]
    }).dropna()

    try:
        cph = CoxPHFitter()
        cph.fit(df, duration_col="time", event_col="event")

        p = cph.summary.loc[col, "p"]
        pvals.append(p)
        valid_cols.append(col)

    except:
        pvals.append(1)
        valid_cols.append(col)

fdr = multipletests(pvals, method="fdr_bh")[1]

significant_nodes = [col for col, f in zip(valid_cols, fdr) if f < 0.05]
print("Significant nodes:", len(significant_nodes))

In [ ]:
latent_sig = latent_df[significant_nodes]

scaler = StandardScaler()
X = scaler.fit_transform(latent_sig)

k_range = range(2, 8)
sil_scores = []
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X)
    sil_scores.append(silhouette_score(X, labels))

k_vals = list(k_range)
best_k = k_vals[np.argmax(sil_scores)]
best_score = max(sil_scores)

print("Best K:", best_k)

plt.figure(figsize=(6, 4))

plt.plot(k_vals, sil_scores, marker='o')
plt.axvline(best_k, linestyle='--', alpha=0.7)
plt.scatter(best_k, best_score, s=80)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Optimal Cluster Selection")

plt.xticks(k_vals)
plt.grid(alpha=0.2)

plt.tight_layout()
plt.show()

Identification of Recurrence-Risk Subgroups


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
latent_sig_scaled = scaler.fit_transform(latent_sig)

kmeans = KMeans(n_clusters=2, random_state=42, n_init=20)
clusters = kmeans.fit_predict(latent_sig_scaled)

print(pd.Series(clusters).value_counts())

In [ ]:
risk_df = survival.copy()
risk_df["cluster"] = clusters

group0_mean = risk_df[risk_df["cluster"] == 0]["time"].mean()
group1_mean = risk_df[risk_df["cluster"] == 1]["time"].mean()

if group0_mean > group1_mean:
    label_map = {0: "Low Risk", 1: "High Risk"}
else:
    label_map = {1: "Low Risk", 0: "High Risk"}

print("Cluster labels:", label_map)

kmf = KaplanMeierFitter()
plt.figure(figsize=(7,5))

for label in ["Low Risk", "High Risk"]:
    for c, name in label_map.items():
        if name == label:
            mask = risk_df["cluster"] == c

            kmf.fit(
                durations=risk_df["time"][mask],
                event_observed=risk_df["recurrence"][mask],
                label=name
            )

            kmf.plot_survival_function(ci_show=True)

g0 = risk_df[risk_df["cluster"] == 0]
g1 = risk_df[risk_df["cluster"] == 1]

lr = logrank_test(
    g0["time"], g1["time"],
    event_observed_A=g0["recurrence"],
    event_observed_B=g1["recurrence"]
)

p_value = lr.p_value
print("Log-rank p-value:", p_value)

plt.xlabel("Time (Days)")
plt.ylabel("Survival Probability")
plt.title("Kaplan–Meier Survival Analysis")

plt.legend(frameon=False)

plt.text(
    0.6, 0.3,
    f"Log-rank p = {p_value:.2e}",
    transform=plt.gca().transAxes,
    fontsize=10
)
plt.xlim(0, 4000)

plt.grid(alpha=0.08)
plt.tight_layout()

plt.savefig("KM_plot.png", dpi=400)

plt.show()